# 04 – Queries across both stars (shared `dim_product`)
Needs `fact_sales` with ProductID, OrderDate, OrderQty, UnitPrice (+ `dim_product`) in the same schema.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA silver;

## Query 1 – 5 products with the thinnest margin (avg sales price − avg purchase price)

In [0]:
%sql
WITH s AS (SELECT ProductKey AS ProductID, AVG(Unit_Price) AS avg_sales_price FROM workspace.default.fact_sales GROUP BY ProductKey),
     p AS (SELECT ProductID, AVG(UnitPrice) AS avg_purchase_price FROM fact_purchases GROUP BY ProductID)
SELECT dp.Product AS product, s.ProductID,
       ROUND(s.avg_sales_price, 2)    AS avg_sales_price,
       ROUND(p.avg_purchase_price, 2) AS avg_purchase_price,
       ROUND(s.avg_sales_price - p.avg_purchase_price, 2) AS margin,
       ROUND((s.avg_sales_price - p.avg_purchase_price) / s.avg_sales_price * 100, 1) AS margin_pct
FROM s JOIN p ON s.ProductID = p.ProductID
LEFT JOIN workspace.default.dim_product dp ON dp.ProductKey = s.ProductID
ORDER BY margin ASC
LIMIT 5

product,ProductID,avg_sales_price,avg_purchase_price,margin,margin_pct
"Half-Finger Gloves, M",465,14.61,49.70,-35.09,-240.1
"Half-Finger Gloves, S",462,14.13,47.68,-33.55,-237.5
"Half-Finger Gloves, S",463,14.66,47.52,-32.86,-224.2
"Half-Finger Gloves, M",464,14.12,45.42,-31.31,-221.8
"Half-Finger Gloves, L",466,14.13,45.42,-31.29,-221.5


## Query 2 – units sold vs units purchased per product, by month

In [0]:
%sql
WITH s AS (SELECT ProductKey AS ProductID, date_trunc('month', OrderDate) AS month, SUM(Quantity) AS units_sold
           FROM workspace.default.fact_sales GROUP BY 1, 2),
     p AS (SELECT ProductID, date_trunc('month', OrderDate) AS month, SUM(OrderQty) AS units_purchased
           FROM fact_purchases GROUP BY 1, 2)
SELECT COALESCE(s.ProductID, p.ProductID) AS ProductID,
       COALESCE(s.month, p.month)         AS month,
       COALESCE(units_sold, 0)            AS units_sold,
       COALESCE(units_purchased, 0)       AS units_purchased,
       COALESCE(units_sold, 0) - COALESCE(units_purchased, 0) AS sold_minus_purchased
FROM s FULL OUTER JOIN p ON s.ProductID = p.ProductID AND s.month = p.month
ORDER BY sold_minus_purchased DESC

ProductID,month,units_sold,units_purchased,sold_minus_purchased
471,2019-08-01T00:00:00.000Z,498,0,498
471,2019-09-01T00:00:00.000Z,464,0,464
491,2019-08-01T00:00:00.000Z,444,0,444
474,2019-08-01T00:00:00.000Z,427,0,427
471,2020-05-01T00:00:00.000Z,422,0,422
491,2019-09-01T00:00:00.000Z,420,0,420
225,2019-08-01T00:00:00.000Z,413,0,413
470,2018-08-01T00:00:00.000Z,409,0,409
234,2019-08-01T00:00:00.000Z,404,0,404
483,2019-09-01T00:00:00.000Z,397,0,397


## Products that sell faster than we restock (total over all months)

In [0]:
%sql
SELECT s.ProductID, dp.Product, s.units_sold, COALESCE(p.units_purchased, 0) AS units_purchased
FROM (SELECT ProductKey AS ProductID, SUM(Quantity) AS units_sold FROM workspace.default.fact_sales GROUP BY ProductKey) s
JOIN (SELECT ProductID, SUM(OrderQty) AS units_purchased FROM fact_purchases GROUP BY ProductID) p
  ON s.ProductID = p.ProductID
LEFT JOIN workspace.default.dim_product dp ON dp.ProductKey = s.ProductID
WHERE s.units_sold > p.units_purchased
ORDER BY s.units_sold - p.units_purchased DESC

ProductID,Product,units_sold,units_purchased
471,"Classic Vest, S",3696,42
470,"Full-Finger Gloves, L",3365,42
491,"Short-Sleeve Classic Jersey, XL",3159,303
474,"Women's Mountain Shorts, S",2729,42
469,"Full-Finger Gloves, M",2197,42
458,"Women's Tights, L",2112,60
456,"Women's Tights, S",2056,60
490,"Short-Sleeve Classic Jersey, L",2217,300
472,"Classic Vest, M",1911,42
465,"Half-Finger Gloves, M",1751,45
